# The Missing Hedge — Result Inspection

This notebook reads a saved, hash-checked paper bundle. It does **not** fetch inputs, optimize hedges, or reproduce cash accounting. The executed copy and HTML are generated by `scripts/reproduce_hedge_paper.py`.

**Conditional research, not executable performance evidence.** Deliverables, execution timestamps and independent distribution validation remain unresolved. Recorded OI is not depth; overlapping historical windows are not independent observations.

In [ ]:
import json
import os
from pathlib import Path
import sys

ROOT = next(parent for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (parent / 'study_config.json').is_file())
sys.path.insert(0, str(ROOT))
from IPython.display import Image, Markdown, display
from src.reporting.hedge_design import identified_directory, load_bundle

BUILD_ID = os.environ.get('HEDGE_PAPER_BUILD', 'legacy-layout-20260909')
manifest, tables = load_bundle(ROOT, BUILD_ID)
bundle = identified_directory(ROOT / 'results/hedge_paper', BUILD_ID)
metrics = json.loads((bundle / 'metrics.json').read_text())
display(Markdown(f"**Build:** `{BUILD_ID}` · **Scientific run:** `{manifest['run_id']}`\n\n"
                 f"{metrics['chain_rows']:,} chain rows · {metrics['etfs']} ETFs · "
                 f"{metrics['snapshots']} snapshots · {metrics['matched_dates']} common outcome dates.\n\n"
                 "Saved output hashes verified. Use the CLI `--verify` for full input/code/dependency checks."))

def figure(name):
    display(Image(filename=str(bundle / 'figures' / f'{name}.png'), width=1000))

## 1. Market anatomy is not executable capacity

The chart shows **all strikes and expirations** on the pilot date, not just eligible hedges. OI is a stock; do not sum repeated dates as trading volume or infer investor motive from unsigned call/put OI. Missing/invalid OI is distinct from zero.

In [ ]:
figure('market_anatomy')
anatomy = tables['market_anatomy']
display(anatomy.loc[(anatomy.snap_date == '2025-01-02') & anatomy.ticker.isin(['LQD', 'TLT', 'IEF'])])

## 2. Cost/protection frontiers

Fixed $100M LQD; January 2–31, 2025; identical joint scenarios across direct, Treasury-substitute and combined menus. These are **design-model CVaRs**, not realized performance. Budget includes ask premium and entry fee; the underlying portfolio is not sold. Financed entry cost is counted once. Uncapped is optimistic; 1%, 5%, 10% OI limits are imposed sensitivities.

Change the selectors below to inspect saved combinations; no solver runs.

In [ ]:
figure('nominal_frontiers')
TAIL_LEVEL = 0.90
BUDGET_BPS = 50
frontiers = tables['frontiers']
selected = frontiers.loc[(frontiers.tail_level == TAIL_LEVEL) & (frontiers.budget_bps == BUDGET_BPS)]
display(selected[['menu', 'method', 'access', 'nominal_cvar_bps', 'worst_model_cvar_bps',
                  'spent_bps', 'n_selected', 'min_strike_spot', 'max_strike_spot', 'verified']])

## 3. Keep strikes and quantities visible

Candidate moneyness differs substantially by ETF. The near-ATM direct menu is empty in the separate 0.95–1.05 sensitivity. An empty menu is not evidence of a synthetic direct hedge. Positions below are continuous contract quantities, not executable orders; the saved full table retains all cells.

In [ ]:
figure('moneyness')
display(tables['moneyness'])
positions = tables['selected_positions']
display(positions.loc[(positions.tail_level == TAIL_LEVEL) & (positions.budget_bps == BUDGET_BPS)].head(24))

## 4. Nominal versus finite-model robust design

Both policies must be evaluated under **both** measures. Robust design trades historical-model risk for the maximum CVaR of three fixed distributions; it is not protection against every distribution or arbitrary mixture. Stress rules are retrospective research assumptions, not historical preregistration. Effective weight count does not remove dependence between overlapping windows.

In [ ]:
figure('model_tradeoff')
display(tables['design_comparison'])
display(tables['support'])

## 5. Common subsequent outcomes — gains and deteriorations

Every policy uses the same 16 supported dates. Positive loss means a loss; negative means a gain. Robust-minus-nominal above zero means deterioration. These are isolated holding periods, not a compounded trading record. Mixed results and clustered snapshots do not establish superiority or a realized CVaR estimate.

In [ ]:
figure('paired_outcomes')
display(tables['outcomes'][['policy', 'access', 'matched_dates', 'mean_loss_bps',
                            'worst_loss_bps', 'mean_spent_bps', 'beats_unhedged']])
display(tables['subsequent_dates'])

## 6. Quote sensitivity and controls

Quote-rule sweeps keep the original expiration: tighter spreads, added zero-bid offers, and matched moneyness. Full saved tables retain both tails and all OI treatments. The TLT-held control protects a **different portfolio** and must not be ranked as an LQD alternative. The no-cap scale check establishes a model property, not real capacity.

In [ ]:
figure('quote_sensitivity')
quotes = tables['quote_sensitivity']
display(quotes.loc[(quotes.tail_level == TAIL_LEVEL) & (quotes.access == 'Uncapped'),
                  ['quote_rule', 'menu', 'method', 'nominal_cvar_bps', 'worst_model_cvar_bps', 'n_candidates']])
control = tables['tlt_control']
display(control.loc[(control.tail_level == TAIL_LEVEL) & (control.budget_bps == BUDGET_BPS),
                    ['menu', 'access', 'cvar_bps', 'cvar_reduction_bps', 'spent_usd', 'verified']])
display(tables['scale_control'])

## 7. Verification and remaining gates

Scientific runs independently recompute objectives from positions and check constraints, family/menu invariants and two-sided dual perturbations. The reporting producer additionally reconstructs common-date summary statistics from saved outcomes. This notebook only checks artifact hashes and displays results.

Before publication: validate contract deliverables, quote/OI timing and execution assumptions, and issuer distributions independently; expand coverage and assess vintage, integer feasibility and market impact. The incomplete legacy-source audit remains separate. No market-wide missing-market or robust-superiority conclusion follows from this notebook.

In [ ]:
duals = tables['duals']
display(duals.groupby(['method', 'constraint'], as_index=False).agg(
    checks=('verified', 'size'), all_verified=('verified', 'all'),
    max_bracket_error_usd=('bracket_error_usd', 'max')))
display(Markdown(f"Generated paper: `{bundle / 'missing_hedge_draft.md'}`\n\n"
                 f"Verify: `.venv/bin/python scripts/reproduce_hedge_paper.py --build-id {BUILD_ID} --verify`"))